In [1]:
# 녹음된 음성 데이터를 전처리하는 과정입니다.
# 녹음된 음성데이터는 비트메이트_TP01/data/dataset/wav_{speaker}/raw폴더에 있어야합니다.
# raw 폴더 내 음성데이터를 변환해서 wavs 폴더에 저장합니다.

# .m4a .wav 상관 없이 .wav 로 변환해줍니다.
# samplingrate는 24000으로 변경합니다. (finetunning시 24000으로 학습시켜야함)
# 무음 구간은 제거합니다
# 볼륨도 일정하게 조절합니다.

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
from pathlib import Path
import numpy as np
import librosa
import soundfile as sf

# =========================
# 경로 설정
# =========================
SPEAKER = "jhc100"

BASE_DIR = Path("/content/drive/MyDrive/비트메이트_TP01")
DATASET_DIR = BASE_DIR / "data" / "dataset" / f"wav_{SPEAKER}"

INPUT_DIR = DATASET_DIR / "raw"
OUTPUT_DIR = DATASET_DIR / "wavs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_SR = 24000

# =========================
# 파일 리스트
# =========================
audio_files = [
    p for p in INPUT_DIR.iterdir()
    if p.suffix.lower() in [".wav", ".m4a"]
]

if not audio_files:
    raise RuntimeError(f"오디오 파일 없음: {INPUT_DIR}")

print(f"총 파일 수: {len(audio_files)}")

# =========================
# 전처리
# =========================
for input_path in audio_files:
    output_path = OUTPUT_DIR / (input_path.stem + ".wav")

    try:
        # 1. 로드 + mono + resample
        audio, sr = librosa.load(
            str(input_path),
            sr=TARGET_SR,
            mono=True
        )

        # 2. 무음 제거 (앞뒤)
        audio, _ = librosa.effects.trim(
            audio,
            top_db=30
        )

        # 3. normalize (볼륨 일정)
        max_val = np.max(np.abs(audio))
        if max_val > 0:
            audio = audio / max_val

        # 4. 너무 짧은 음성 제거 (노이즈 방지)
        if len(audio) < TARGET_SR * 0.3:  # 0.3초 이하 컷
            print(f"[스킵] 너무 짧음: {input_path.name}")
            continue

        # 5. 저장
        sf.write(str(output_path), audio, TARGET_SR)

        print(f"[완료] {input_path.name} → {output_path.name}")

    except Exception as e:
        print(f"[에러] {input_path.name}: {e}")

총 파일 수: 100
[완료] audio_085.wav → audio_085.wav
[완료] audio_078.wav → audio_078.wav
[완료] audio_073.wav → audio_073.wav
[완료] audio_097.wav → audio_097.wav
[완료] audio_071.wav → audio_071.wav
[완료] audio_081.wav → audio_081.wav
[완료] audio_080.wav → audio_080.wav
[완료] audio_079.wav → audio_079.wav
[완료] audio_096.wav → audio_096.wav
[완료] audio_091.wav → audio_091.wav
[완료] audio_094.wav → audio_094.wav
[완료] audio_077.wav → audio_077.wav
[완료] audio_072.wav → audio_072.wav
[완료] audio_099.wav → audio_099.wav
[완료] audio_098.wav → audio_098.wav
[완료] audio_093.wav → audio_093.wav
[완료] audio_095.wav → audio_095.wav
[완료] audio_076.wav → audio_076.wav
[완료] audio_069.wav → audio_069.wav
[완료] audio_074.wav → audio_074.wav
[완료] audio_047.wav → audio_047.wav
[완료] audio_058.wav → audio_058.wav
[완료] audio_046.wav → audio_046.wav
[완료] audio_067.wav → audio_067.wav
[완료] audio_065.wav → audio_065.wav
[완료] audio_083.wav → audio_083.wav
[완료] audio_051.wav → audio_051.wav
[완료] audio_090.wav → audio_090.wav
[완료] aud

In [4]:
import shutil

# =========================
# ref.wav 생성
# =========================
wavs = sorted([
    p for p in OUTPUT_DIR.iterdir()
    if p.suffix == ".wav"
])

if not wavs:
    raise RuntimeError(f"wavs 폴더에 파일 없음: {OUTPUT_DIR}")

ref_path = DATASET_DIR / "ref.wav"
# 길이 가장 긴 파일을 ref로 사용
source_audio = max(wavs, key=lambda p: p.stat().st_size)

# 덮어쓰기 복사
shutil.copy2(source_audio, ref_path)

print("ref.wav 생성 완료")
print("source:", source_audio.name)
print("target:", ref_path)

ref.wav 생성 완료
source: audio_092.wav
target: /content/drive/MyDrive/비트메이트_TP01/data/dataset/wav_jhc100/ref.wav
